# Study 2: Which design gives useful tradeoffs?

- Notebooks 04-06 suggest that the current reward, mitigation actions, and income may push agents toward frequent high mitigation and little later development.
- I want to understand those parts of the design before trying another set of rules. This notebook starts with the current settings and evidence

## What would a useful setup show?

- **Environmental challenge:** Development without mitigation can fail, while sensible mitigation can prevent failure. Pollution should still vary enough to compare policies.
- **Meaningful choices:** Low, medium, and high prevention and cleanup should have different uses. Paying for a higher level should sometimes do more, and its extra cost should matter.
- **Economic choices:** Development should create a reason to invest over time. Currency should sometimes constrain decisions, rather than accumulate without changing behavior.
- **Fair comparison:** Every proposed setup should be evaluated with the same starts, training settings, and common outcome measures. I will look at pollution, spending, EDI, currency, failures, actions, and variation across training seeds.

## Read the current settings

In [1]:
import pandas as pd

from study_2_economic_sustainability.config import EnvironmentConfig
from study_2_economic_sustainability.environment import (ACTION_NAMES, EconomicSustainabilityEnvironment, action_parts, development_gain)

from study_2_economic_sustainability.evaluation import run_episode
from study_2_economic_sustainability.train import load_q_tables

config = EnvironmentConfig()

### What do environmental investment actions do?

- Prevention reduces new emissions by a fraction. Cleanup removes part of the existing pollution stock, subject to pollutant-specific limits.
- The table shows investment cost, prevention share, cleanup capacity per pollutant, and gross emissions at EDI 0 and 100. Emissions are totals across the four pollutants before prevention.

In [2]:
action_rows = []
for action, name in enumerate(ACTION_NAMES):
    development_choice, investment_choice = action_parts(action)
    investment_cost = 0
    prevention_fraction = 0
    cleanup_capacity = 0

    if investment_choice in (1, 2, 3):
        investment_cost = config.prevention_costs[investment_choice]
        prevention_fraction = config.prevention_fractions[investment_choice]
    elif investment_choice in (4, 5, 6):
        cleanup_level = investment_choice - 3
        investment_cost = config.cleanup_costs[cleanup_level]
        cleanup_capacity = config.cleanup_capacities[cleanup_level]

    emissions_at_edi_0 = 0
    emissions_at_edi_100 = 0
    for pollutant in config.pollutants:
        emissions_at_edi_0 += config.baseline_emissions[pollutant]
        emissions_at_edi_100 += (
            config.baseline_emissions[pollutant]
            + config.edi_emissions_per_point[pollutant] * 100
        )

    total_cost = investment_cost
    if development_choice:
        total_cost += config.development_cost

    action_rows.append({
        "action": name,
        "action_cost": total_cost,
        "prevention_fraction": prevention_fraction,
        "cleanup_capacity_per_pollutant": cleanup_capacity,
        "gross_emissions_at_edi_0": emissions_at_edi_0,
        "gross_emissions_at_edi_100": emissions_at_edi_100,
    })

display(pd.DataFrame(action_rows).set_index("action"))

,action_cost,prevention_fraction,cleanup_capacity_per_pollutant,gross_emissions_at_edi_0,gross_emissions_at_edi_100
action,,,,,
maintain_none,0.0,0.00,0.0,6.5,23.5
develop_none,100.0,0.00,0.0,6.5,23.5
maintain_prevent_low,10.0,0.25,0.0,6.5,23.5
maintain_prevent_medium,25.0,0.50,0.0,6.5,23.5
maintain_prevent_high,50.0,0.75,0.0,6.5,23.5
maintain_clean_low,15.0,0.00,2.0,6.5,23.5
maintain_clean_medium,40.0,0.00,5.0,6.5,23.5
maintain_clean_high,80.0,0.00,10.0,6.5,23.5
develop_prevent_low,110.0,0.25,0.0,6.5,23.5


- Emissions rise with EDI, but each pollutant has its own baseline and development-related rate. Prevention applies the same selected fraction to each pollutant's current emissions; cleanup acts on accumulated stocks.

### When does income cover high environmental investment?

- Income is `20 + 0.5 × EDI` currency per step. High prevention and high cleanup have different costs.
- The income-minus-cost columns describe the net balance change after one purchase. A region must already have enough currency to pay before receiving that step's income.

In [3]:
income_rows = []

for edi in (0, 40, 60, 70, 100):
    income = config.income_base + config.income_per_edi * edi
    income_rows.append({
        "edi": edi,
        "income_per_step": income,
        "income_minus_high_prevention_cost": income - config.prevention_costs[3],
        "income_minus_high_cleanup_cost": income - config.cleanup_costs[3],
        "income_minus_develop_and_prevent": (income - config.development_cost - config.prevention_costs[3]),
        "edi_gain_if_develop": development_gain(edi),
    })

display(pd.DataFrame(income_rows).set_index("edi"))

,income_per_step,income_minus_high_prevention_cost,income_minus_high_cleanup_cost,income_minus_develop_and_prevent,edi_gain_if_develop
edi,,,,,
0,20.0,-30.0,-60.0,-130.0,10
40,40.0,-10.0,-40.0,-110.0,5
60,50.0,0.0,-30.0,-100.0,5
70,55.0,5.0,-25.0,-95.0,2
100,70.0,20.0,-10.0,-80.0,0


- Compare the two high investment costs separately. Income alone may cover high prevention at a different EDI than high cleanup, but the starting balance must cover the purchase.
- The EDI gain from a development purchase falls from 10 to 5 and then 2 as EDI rises. Income above the base level also contributes a smaller ongoing reward.
- Notebook 05 showed that balances grow after development choices fade. This table shows why affordability alone may provide little reason to invest further.

### How large is the pollution penalty?

- This table shows the pollution part of local reward when all pollutant stocks are set to the same selected level. It does not predict the full reward of any action.
- Local economic reward combines EDI gain with the ongoing-income bonus. Currency spent on valid actions is still not directly subtracted from reward.

In [4]:
reward_rows = []
for resulting_load in (0, 10, 20, 40, 60, 100):
    weighted_damage = 0
    for pollutant in config.pollutants:
        normalized_load = resulting_load / config.pollution_limits[pollutant]
        excess = max(0, normalized_load - config.damage_threshold_fraction)
        damage = normalized_load + 4 * excess ** 2
        weighted_damage += config.pollutant_weights[pollutant] * damage

    pollution_reward = -config.pollution_weight * weighted_damage
    reward_rows.append({"resulting_pollution_load": resulting_load, "pollution_reward": pollution_reward})

display(pd.DataFrame(reward_rows).set_index("resulting_pollution_load"))

,pollution_reward
resulting_pollution_load,
0,-0.00
10,-0.10
20,-0.20
40,-0.40
60,-0.64
100,-2.00


## Baseed on the other notebooks though:

- Notebook 04: Original local and mixed agents chose high mitigation on about 85% of region-steps, developed mostly early, and had no failures on the 20 held-out starts.
- Notebook 05: In later stages, medium would have matched 96-97% of paid high actions under local and mixed reward on that single step. High often bought more capacity than was used.
- Notebook 06: Replacing high with medium across whole episodes reduced spending sharply, raised local and mixed GPI modestly, and caused no failures. Training with a 0.02 mitigation-spending reward penalty nearly removed high choices under local and mixed reward, but development choices barely changed. Global results varied more across seeds.

Based on these notebooks and results I have two new questions: 1. When do the agents stop developing and why?  2. Once they have stopped investing in development, what do they do instead?

## When do agents stop developing?

- I'll use the original saved policies, with five training seeds and the same 20 held-out starts as notebooks 04-06.
- For each region in each episode, the last **successful** development action marks the point after which that region never develops again. This is a description of the observed episode, not proof of why the agent stopped.

### Load the original policies

In [5]:
REWARD_MODES = ("local", "global", "mixed")
TRAINING_SEEDS = (11, 23, 37, 41, 53)
EVALUATION_SEEDS = tuple(range(100_000, 100_020))

In [6]:
agents_by_run = load_q_tables(REWARD_MODES, TRAINING_SEEDS, results_directory="../results", notebook_number=3)
print(f"Loaded {len(agents_by_run)} policies")

Loaded 15 policies


### Record each region's decisions

- `steps_remaining` is the number of steps left under the 200-step limit. The agent does not observe this number, and an episode can end earlier if pollution reaches 100.

In [7]:
def region_rows(environment, actions, info, training_seed, environment_seed, step):
    rows = []
    for region in range(4):
        rows.append({
            "reward_mode": environment.reward_mode,
            "training_seed": training_seed,
            "environment_seed": environment_seed,
            "region": region,
            "step": step,
            "steps_remaining": config.episode_steps - step,
            "action": actions[region],
            "action_name": info["action_names"][region],
            "valid": info["valid_actions"][region],
            "edi_gain": info["development_gains"][region],
            "edi": info["development_indices"][region],
            "currency": info["currency_balances"][region],
            "regional_pollution": info["pollution_loads"][region],
            "gpi": info["global_pollution_index"],
        })
    return rows

In [8]:
def collect_episode(agents, reward_mode, training_seed, environment_seed):
    environment = EconomicSustainabilityEnvironment(reward_mode)
    transitions, terminated, truncated = run_episode(environment, agents, environment_seed)
    rows = []

    for transition in transitions:
        rows.extend(region_rows(
            environment,
            transition["actions"],
            transition["info"],
            training_seed,
            environment_seed,
            transition["step"],
        ))

    return rows

In [9]:
step_rows = []
for reward_mode in REWARD_MODES:
    for training_seed in TRAINING_SEEDS:
        agents = agents_by_run[(reward_mode, training_seed)]
        for environment_seed in EVALUATION_SEEDS:
            step_rows.extend(collect_episode(agents, reward_mode, training_seed, environment_seed))

steps = pd.DataFrame(step_rows)
print(f"Recorded {len(steps):,} region-steps")

Recorded 240,000 region-steps


### Find the last development action

- A successful development action increases EDI. I take the last such action for each region and episode.
- Regions that never develop are counted separately. Regions that end at EDI 100 have reached the development limit, so their later lack of development has a different explanation.

In [ ]:
RUN_KEYS = ["reward_mode", "training_seed", "environment_seed", "region"]

In [10]:
last_development = (steps[steps["edi_gain"] > 0].sort_values("step").drop_duplicates(RUN_KEYS, keep="last").copy())
last_development["at_edi_cap"] = last_development["edi"] >= 100
last_development["can_afford_development"] = ((last_development["currency"] >= config.development_cost) & last_development["at_edi_cap"])

region_episodes = steps.loc[steps["step"] == 1, RUN_KEYS]

In [ ]:
run_status = region_episodes.merge(last_development[RUN_KEYS + ["step"]], on=RUN_KEYS, how="left")
run_status["never_developed"] = run_status["step"].isna()

In [11]:
never_by_seed = (run_status.groupby(["reward_mode", "training_seed"])["never_developed"].mean() * 100)
never_by_mode = never_by_seed.groupby(level="reward_mode").agg(["mean", "std"])

display(never_by_mode.rename(columns={"mean": "mean_percent", "std": "seed_sd"}).round(1))

,mean_percent,seed_sd
reward_mode,,
global,0.0,0.0
local,0.0,0.0
mixed,0.0,0.0


- This table shows the share of region-episodes with no successful development at all. The mean and standard deviation are across five training seeds.

In [12]:
stop_by_seed = last_development.groupby(["reward_mode", "training_seed"]).agg(
    last_development_step=("step", "mean"),
    steps_remaining=("steps_remaining", "mean"),
    edi=("edi", "mean"),
    currency=("currency", "mean"),
    regional_pollution=("regional_pollution", "mean"),
    gpi=("gpi", "mean"),
    at_edi_cap_percent=("at_edi_cap", "mean"),
    can_afford_development_percent=("can_afford_development", "mean"),
)

last_development_step       steps_remaining         edi      
                             mean   std            mean   std  mean   std
reward_mode                                                              
global                       26.0   6.0           174.0   6.0  73.2   3.2
local                        32.7  14.9           167.3  14.9  79.0  12.9
mixed                        35.5  11.4           164.5  11.4  81.9   8.2

In [ ]:
stop_by_seed[["at_edi_cap_percent", "can_afford_development_percent"]] *= 100
display(stop_by_seed.groupby(level="reward_mode")[["last_development_step", "steps_remaining", "edi"]].agg(["mean", "std"]).round(1))

- `last_development_step` is the final successful development step, averaged within each seed and then across seeds.

In [13]:
display(stop_by_seed.groupby(level="reward_mode")[["currency", "regional_pollution", "gpi", "at_edi_cap_percent", "can_afford_development_percent"]].mean().round(1))

,currency,regional_pollution,gpi,at_edi_cap_percent,can_afford_development_percent
reward_mode,,,,,
global,192.5,13.4,13.0,6.2,92.0
local,216.7,15.1,14.5,22.5,74.2
mixed,209.7,16.3,16.2,30.2,69.8


- These values describe the recorded state after the last development step, including that step's income.
- `can_afford_development_percent` means the region has at least 100 currency for another purchase and its EDI is below 100.
- If many regions stop while they can still afford development, lack of currency alone is unlikely to explain the behavior. High pollution at that point might support an environmental-risk explanation, but the table cannot establish causation.

### What do they choose afterward?

- I count actions from the step after the final successful development purchase until that region's episode ends. Regions that never developed are excluded from this table.

In [14]:
stops = last_development[RUN_KEYS + ["step"]].rename(columns={"step": "last_development_step"})
after = steps.merge(stops, on=RUN_KEYS, how="inner")
after = after[after["step"] > after["last_development_step"]].copy()

action_name,maintain_none,develop_none,maintain_prevent_low,maintain_prevent_medium,maintain_prevent_high,maintain_clean_low,maintain_clean_medium,maintain_clean_high,develop_prevent_low,develop_prevent_medium,develop_prevent_high,develop_clean_low,develop_clean_medium,develop_clean_high
reward_mode,,,,,,,,,,,,,,
global,11.8,0.0,26.2,14.3,12.6,19.4,8.1,7.5,0.0,0.0,0.1,0.0,0.0,0.0
local,7.9,3.3,10.6,19.6,16.9,10.6,17.6,9.0,0.1,0.0,0.0,0.6,3.8,0.0
mixed,20.1,0.2,11.4,19.7,16.0,13.5,11.7,6.6,0.6,0.0,0.1,0.0,0.1,0.1


In [ ]:
after_counts = (after.groupby(["reward_mode", "training_seed", "action_name"]).size().unstack(fill_value=0).reindex(columns=ACTION_NAMES, fill_value=0))

after_percent = after_counts.div(after_counts.sum(axis=1), axis=0) * 100
display(after_percent.groupby(level="reward_mode").mean().round(1))

- Each row shows the mean share of the 14 combined actions across five training seeds after development stops. Action names distinguish whether a region maintains or develops while choosing prevention or cleanup.

In [15]:
after["invalid"] = ~after["valid"]
invalid_by_seed = (after.groupby(["reward_mode", "training_seed"])["invalid"].mean() * 100)
display(invalid_by_seed.groupby(level="reward_mode").agg(["mean", "std"]).round(1))

,mean,std
reward_mode,,
global,0.2,0.3
local,7.8,9.5
mixed,1.1,1.0


- This is the percentage of post-development choices the environment rejected, averaged across seeds. A high value would mean the chosen-action table needs extra care when describing actual spending.